<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [1]:
import os
import json
import pandas as pd
import pip
import string
import re
import numpy as np

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [2]:
#! kaggle datasets download -d gsimonx37/letterboxd

We only consider a subset of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [3]:
DATA_DIR = "./letterboxd"
# members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
# with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
#     for file_name in members_to_extract:
#         zip_ref.extract(file_name + '.csv', DATA_DIR)
!tar xf ./drive/MyDrive/letterboxd.tar.gz

We then prepare the entry point for the Spark functionalities that will we use from now on.

In [4]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf ./drive/MyDrive/spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

In [5]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "./drive/MyDrive/spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form.

In [6]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

def extract_data(member):
    """ Extract and preprocess data from a specified CSV file

    Args:
        member (str): The category of data to be extracted. Expected values are file names such as 'actors'
                      which correspond to CSV files in the DATA_DIR.

    Returns:
        pyspark.RDD: An RDD where each element is a tuple. The first element is the record ID and the second
                     is a dictionary mapping column names to their corresponding values.
    """
    rdd = sc.textFile(DATA_DIR + "/" + member + ".csv")

    if member == 'actors':
        rdd = rdd.zipWithIndex().map(lambda r: r[0] + ',' + str(r[1]))
        rdd = rdd.map(lambda r: re.sub(r'(\d+),[\s\t]+([a-zA-Z])', r'\1,\2', r))
    rdd = rdd.map(lambda r: re.split(r',(?! )', r))  #split only on commas that are followed by a character to avoid splitting sentences e.g. in movie descriptions

    # get column name from csv different from 'id'
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    if member == 'actors':
        column_names[-1] = 'movie_n'
    rdd = (rdd
            .map(lambda r: (r[0], dict(zip(column_names, r[1:]))))
            .filter(lambda r: r[0]!='id'))
    return rdd

for member in members_to_extract:
    letterboxd_RDDs[member] = extract_data(member)


In [7]:
letterboxd_RDDs['themes'] = letterboxd_RDDs['themes'].mapValues(lambda x: {**x, 'theme': x['theme'].strip('"')})

Let's look at the amount of rows for each member.

In [8]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Number of rows for actors:	5798450
Number of rows for crew:	4720183
Number of rows for genres:	1046849
Number of rows for movies:	941597
Number of rows for themes:	125641


A glimpse at the structure of the rows in the RDDs of each member.

In [9]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 ('1000001', {'name': 'Margot Robbie', 'role': 'Barbie', 'movie_n': '1'})
Row for crew:	 ('1000001', {'role': 'Director', 'name': 'Greta Gerwig'})
Row for genres:	 ('1000001', {'genre': 'Comedy'})
Row for movies:	 ('1000001', {'name': 'Barbie', 'date': '2023', 'tagline': "She's everything. He's just Ken.", 'description': '"Barbie and Ken are having the time of their lives in the colorful and seemingly perfect world of Barbie Land. However, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans."', 'minute': '114', 'rating': '3.86'})
Row for themes:	 ('1000001', {'theme': 'Humanity and the world around us'})


Below we have prepared a function to extract a sample of the data, based on the ids in the datasets. The maximum size of the sample is $125641$.

In [10]:
def get_sample(rdd, size):
    """ Extract a sample of records from the RDD based on a specified size

    Args:
        rdd (pyspark.RDD): The input RDD containing records, where each record's first element is expected
                           to be an ID as a string.
        size (int): The desired number of records to sample. The function filters records with IDs less
                    than or equal to 1,000,000 plus the specified size.

    Returns:
        pyspark.RDD: An RDD containing the filtered sample of records.
    """
    return rdd.filter(lambda r: int(r[0],10)<=1000000+size)


sample_size = 100
for member in members_to_extract:
    letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], sample_size)

For each member the available attributes are the following:

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genres                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | themes                              |

</br>

For this project we would like to focus on the following features:
<a name="table1"></a>

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 10 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genres                               |
| **movies**   | name, date, minute, rating |
| **themes**   | themes                               |


We keep only the `n_actors` most relevant actors in each movie.

In [11]:
n_actors = 6
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .groupByKey().map(lambda r: (r[0], list(r[1])))
                            .map(lambda r: (r[0], sorted(r[1], key=lambda x: x["movie_n"])[:n_actors]))
                            .map(lambda r: (r[0], list_dicts_to_dict(r[1]))))

We set up some primitives to manipulate the dictionaries that are the values in the RDDs' rows.

In [12]:
def filter_dict_fields(d, final_fields):
    """ Filter a dictionary to retain only specified fields

    Args:
        d (dict): The input dictionary from which fields need to be filtered.
        final_fields (list): A list of keys representing the fields to retain in the dictionary.

    Returns:
        dict: A new dictionary containing only the key-value pairs where the key is in `final_fields`.
    """
    return {key: d[key] for key in final_fields if key in d}

def list_dicts_to_dict(l):
    """ Convert a list of dictionaries into a dictionary of lists

    Args:
        l (list): A list of dictionaries, where each dictionary contains the same keys.

    Returns:
        dict: A dictionary where each key corresponds to a list of values extracted from the
              dictionaries in l.
    """

    return {key: [d[key] for d in l] for key in l[0]}

def remove_dict_field(d, field):
    """ Remove a specified field from a dictionary

    Args:
        d (dict): The input dictionary from which the field should be removed.
        field (str): The key of the field to be removed from the dictionary.

    Returns:
        dict: The updated dictionary with the specified field removed.

    Raises:
        KeyError: If the specified field is not found in the dictionary.
    """

    del d[field]
    return d

def rename_key_in_dict(d, old_key, new_key):
    """ Rename a key in a dictionary

    Args:
        d (dict): The input dictionary where a key needs to be renamed.
        old_key (str): The existing key in the dictionary that needs to be renamed.
        new_key (str): The new key name to replace the old key.

    Returns:
        dict: The updated dictionary with the key renamed.

    Raises:
        KeyError: If the old key is not found in the dictionary.
    """
    d[new_key] = d.pop(old_key)
    return d

def list_to_dict(l, field_base_name):
    """ Convert a list into a dictionary with keys based on a specified base name

    Args:
        l (list): The input list whose elements are to be converted into dictionary values.
        field_base_name (str): The base string used to construct dictionary keys. Keys will be
                               generated as `field_base_name1`, `field_base_name2`, etc.

    Returns:
        dict: A dictionary where keys are generated from the base name and the index of each list
              element, and values are the corresponding elements from the list.
    """
    return {f"{field_base_name}{i + 1}": value for i, value in enumerate(l)}

We only keep the actors' names, and create a field for each of the `n_actors` selected actors.

In [13]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                            .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'actors'))))

In [14]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors'].map(lambda r: (r[0], {**r[1],**list_to_dict(r[1]['actors'], 'actor')}))
                            .map(lambda r: (r[0], remove_dict_field(r[1], 'actors'))))

For example,

In [15]:
letterboxd_RDDs['actors'].first()

('1000004',
 {'actor1': 'Edward Norton',
  'actor2': 'Brad Pitt',
  'actor3': 'Helena Bonham Carter',
  'actor4': 'Meat Loaf',
  'actor5': 'Jared Leto',
  'actor6': 'Zach Grenier'})

We filter only the directors from the crew dataset, and we only keep their name.

In [16]:
letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                           .filter(lambda r: r[1]['role']=='Director')
                           .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                           .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'director'))))

For each movie we only store the attributes listed in the <a href="#table1">table above</a>.

In [17]:
letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name', 'date', 'minute', 'rating']))))

For each category we account for movies having multiple values for a given attribute.

In [18]:
for member in members_to_extract:
    letterboxd_RDDs[member] = (letterboxd_RDDs[member]
                            .groupByKey().map(lambda r: (r[0], list(r[1])))
                            .map(lambda r: (r[0], list_dicts_to_dict(r[1]))))

In [19]:
letterboxd_RDDs['crew'].take(5)

[('1000012', {'director': ['Damien Chazelle']}),
 ('1000017', {'director': ['Christopher Nolan']}),
 ('1000021',
  {'director': ['Joaquim Dos Santos', 'Justin K. Thompson', 'Kemp Powers']}),
 ('1000029', {'director': ['Michel Gondry']}),
 ('1000036', {'director': ['Quentin Tarantino']})]

In [20]:
movies_RDD = letterboxd_RDDs[members_to_extract[0]]
for member in members_to_extract[1:]:
    movies_RDD = movies_RDD.join(letterboxd_RDDs[member]).mapValues(lambda x: {**x[0], **x[1]})

movies_RDD = (movies_RDD.flatMap(lambda x: [((x[0], key), value) for key, value in x[1].items()])
                .map(lambda r: ((int(r[0][0]), r[0][1]), r[1])))

A generic row of `movies_RDD` has the following format:
<p align=center><i>((id, category), list_of_values)</i></p>

For example,

In [21]:
movies_RDD.take(5)

[((1000064, 'actor1'), ['Amy Adams']),
 ((1000064, 'actor2'), ['Jeremy Renner']),
 ((1000064, 'actor3'), ['Forest Whitaker']),
 ((1000064, 'actor4'), ['Michael Stuhlbarg']),
 ((1000064, 'actor5'), ['Tzi Ma'])]

# Data pre-processing

To preserve the independent role of each attribute we have decided to measure their similarity using cosine distance. This entails translating all data regarding a movie into a vector with components in $\mathbb{R}$.

Below we list the data types of the various attributes.

| **Attributes**    | **Datatype**                       |
|--------------|-----------------------------------------|
| **actors**    | text     |
| **director**     | text                        |
| **genre**   | text                             |
| **theme**   | text                             |
| **name**   | text |
| **date**   | numerical |
| **minute**   | numerical |
| **rating**   | numerical                             |

It is evident that the textual attributes have to be transformed into values to be able to work in an Euclidean space. The following paragraphs are dedicated to these transformations. We refer to the report for a discussion regarding the chosen methods.


To simplify the operations that follow we chang the RDD structure such that a row has the following format:
<p align=center><i>((id, category), list_of_values)</i></p>

## Text encoding

We bring all the strings in data dictionary to lower case, eccept for the names of actors and director. In addition to names always being capitalized, we will not consider them from a semantic point of view, so their uniformation in preparation for the next steps would be useless.

In [22]:
categories_all = [f'actor{i+1}' for i in range(n_actors)]+['director', 'genre', 'name', 'theme', 'date', 'minute', 'rating']

In [23]:
movies_hashed_RDD = movies_RDD
movies_RDD = movies_RDD.map(lambda r: (r[0][0], (r[0][1], r[1]))).groupByKey().mapValues(list)

In [24]:
def apply_to_categories(rdd, l, f):
    """ Apply a function to specific categories within an RDD

    Args:
        rdd (pyspark.RDD): The input RDD containing tuples, where the first element is a key (typically
                           a tuple or a string) and the second element is a list of values.
        l (list): A list of categories (keys) to which the function should be applied.
        f (function): A function to apply to each element of the lists in the second element of the tuples
                      for the specified categories.

    Returns:
        pyspark.RDD: An RDD where the function `f` has been applied to the lists in the second element of
                     the tuples for the specified categories. Other tuples remain unchanged.
    """
    return rdd.map(lambda r: (r[0], [f(s) for s in r[1]]) if r[0][1] in l else r)

In [25]:
categories_lower = ['genre', 'name', 'theme']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_lower, lambda x: x.lower())

To distill the semantics of the movie's theme we apply the following NLP processing steps:
* remove stop words
* replace the words with their lemmatized version

In [26]:
import spacy
! python -m spacy download en_core_web_md -q

nlp = spacy.load("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 20.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [27]:
from functools import reduce

combine_functions = lambda *funcs: lambda x: reduce(lambda v, f: f(v), funcs, x) #v accumulator value
remove_punctuation = lambda x: re.sub(r'[^\w\s]','',x)
remove_multiple_spaces = lambda x: re.sub(r'\s+',' ',x)
remove_stop_words_func = lambda x: " ".join([token.text for token in nlp(x) if not token.is_stop])
lemmatize_func = lambda x: " ".join([token.lemma_ for token in nlp(x)])

nlp_processing = combine_functions(remove_punctuation, remove_multiple_spaces, remove_stop_words_func, lemmatize_func)

categories_nlp = ['theme']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_nlp, nlp_processing)

### Embedding strings using word2vec

We apply word2vec embeddings to attributes:
* name
* genres
* themes

In [28]:
embedding_func = lambda x: nlp(x).vector

categories_word2vec = ['name', 'genre', 'theme']
movies_RDD = apply_to_categories(movies_RDD, categories_word2vec, embedding_func)

For example a value for the *name* attribute will result as such:

In [30]:
id, value = movies_hashed_RDD.filter(lambda r: r[0][1]=='name').first()
print("id:\t {}\nvalue:\t {}".format(id,value[0]))

id:	 (1000098, 'name')
value:	 wall·e


### Hashing strings

We hash the strings of the features:
* actors
* director
person name hash
(no bias towards similar names)

In [ ]:
import math

def hash_object(byte_obj, seed, hash_bucket_size):
    """ Hash an object into a bucket of values [0,hash_bucket_size-1] on the basis of the seed

        Args:
            byte_obj (byte array): an object in byte format
            seed (int): seed for the hash function
            hash_bucket_size (int): the size of the bucket to which the objects get hashed to

        Returns:
            int: hashed object
        """
    m=hashlib.shake_256()
    m.update(byte_obj)
    m.update(bytes(seed))
    required_bytes = math.ceil(hash_bucket_size / 8)
    hash_output = m.digest(required_bytes)
    hashed_value = int.from_bytes(hash_output, 'little')
    return hashed_value % hash_bucket_size

In [ ]:
import hashlib
hash_func = lambda v: hash_object(bytearray(v,'utf-8'),42,5*256)

categories_hash = [f"actor{i+1}" for i in range(n_actors)]+['director']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_hash, hash_func)

For example a value for the *director* attribute will result as such:

In [ ]:
id, value = movies_hashed_RDD.filter(lambda r: r[0][1]=='director').first()
print("id:\t {}\nvalue:\t {}".format(id,value[0]))

## Vector preparation

Now that we have prepared all categories in a targeted way, we can proceed by building the vectors of the movies.

**Reduce attributes with multiple values**

We simply average the values or arrays for attributes that have multiple values.

In [ ]:
vectors_RDD = movies_hashed_RDD.map(lambda r: (r[0], ((np.mean(r[1], axis=0)).tolist() if r[0][1] in categories_word2vec else [np.mean([float(s) for s in r[1]])])))

**Normalization & PCA**

We reduce the amount of features for each vector to avoid suffering from the curse of dimensionality when evaluating their similarity. Before reducing the vectors we standardize their components such that each feature has a mean of $0$ and a standard deviation of $1$.

In [ ]:
vectors_per_cat_RDD = (vectors_RDD.map(lambda r: (r[0][1], (r[0][0], r[1])))
                    .map(lambda r: (r[0], [((r[1][1][i],r[1][0]),i) for i in range(len(r[1][1]))]))
                    .flatMap(lambda r: [(r[0],el) for el in r[1]])
                    .map(lambda r: ((r[0],r[1][1]), r[1][0])))

In [ ]:
features_per_cat_RDD = (vectors_per_cat_RDD.map(lambda r: (r[0], r[1][0]))
                                            .groupByKey().mapValues(list))
mean_cat_RDD = features_per_cat_RDD.map(lambda r: (r[0], np.mean(r[1])))
mean_cat_dict = dict(mean_cat_RDD.collect())
std_cat_RDD = features_per_cat_RDD.map(lambda r: (r[0], np.std(r[1])))
std_cat_dict = dict(std_cat_RDD.collect())

In [ ]:
mean_cat_dict_br = sc.broadcast(mean_cat_dict)
std_cat_dict_br = sc.broadcast(std_cat_dict)
vectors_stand_RDD = (vectors_per_cat_RDD.map(lambda r: (r[0], (r[1][1],(r[1][0]-mean_cat_dict_br.value[r[0]])/std_cat_dict_br.value[r[0]])))
                    .map(lambda r: (r[1][0], (r[0], r[1][1])))
                    .groupByKey().mapValues(list)
                    .mapValues(sorted)
                    .map(lambda r: (r[0], [v[1] for v in r[1]])))

We build the covariance matrix for the data.

In [ ]:
n_vectors = vectors_stand_RDD.count()

def outer_prod(v):
    """ Compute the outer product of a vector with itself

    Args:
        v (list or numpy.ndarray): A vector (list or NumPy array) whose outer product with itself is to be computed.

    Returns:
        numpy.ndarray: A 2D NumPy array representing the outer product of the input vector.
    """
    return np.outer(np.array(v), np.array(v))

cov_matrix = (vectors_stand_RDD.map(lambda r: (r[0], outer_prod(r[1])))
                .reduce(lambda a, b: (1, a[1] + b[1])))[1] / n_vectors

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Sort eigenvalues and eigenvectors in descending order
sorted_indices = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues = eigenvalues[sorted_indices]
sorted_eigenvectors = eigenvectors[:, sorted_indices]

We only retain the components such that their total cumulative explained variance is at least $95\%$.

In [ ]:
explained_variance = sorted_eigenvalues / np.sum(sorted_eigenvalues)

cumulative_explained_variance = np.cumsum(explained_variance)

threshold = 0.95
num_components = np.argmax(cumulative_explained_variance >= threshold) + 1

print(f"Number of components explaining at least {threshold*100}% of the variance: {num_components}")

In [ ]:
k = num_components
top_k_eigenvectors = sorted_eigenvectors[:, :k]

In [ ]:
vectors_reduced_RDD = (vectors_stand_RDD.map(lambda r: (r[0], np.dot(top_k_eigenvectors.T, r[1]))))

An example of reduced vector:

In [ ]:
id, vector = vectors_reduced_RDD.first()
print("movie id:\t {}\n  vector:\t {}".format(id,vector))

## Locality Sensitive Hashing (LSH) technique

### Locality sentitive family for cosine distance

We build the locality sensitive family $\mathbf{F}$ as a set of randomly chosen vectors $\{v_{f\in\mathbf{F}}\}$. Given two vectors $x$ and $y$, they make a candidate pair of similar items if and only if the dot products $x\cdot v_f$ and $x \cdot v_f$ have the same sign. A family of functions $\mathbf{F}$ built as described is a locality-sensitive family for the cosine distance.

We will also refer to the random vectors in $\mathbf{F}$ as hash functions.

Since we will compute the hash functions of each of the elements in the dataset, given each of the hash functions in $\mathbf{F}$, we try to simplify the computation of the cosine distance between vectors. We do so by restricting the random choice of vectors to those having components $+1$ or $-1$. Hence the dot product of any vector $x$ with a vector in such a family $\mathbf{F}$ is given by its algebraic sum $x$'s components, where the signs depend on the components of the random vector.

In [ ]:
signature_length=200
hash_funcs_RDD = (sc.parallelize([(i,1) for i in range(signature_length)])
                    .map(lambda r: (r[0],np.random.choice([-1, 1], num_components))))

For each of the vectors in $\mathbf{F}$ (called `hash_funcs_RDD` in the code), we compute the dot products with each of the (reduced) vectors in the dataset.

In [ ]:
vectors_hash_RDD = (vectors_reduced_RDD.cartesian(hash_funcs_RDD)
                    .map(lambda r: ((r[0][0],r[1][0]), (r[0][1], r[1][1]))))

In [ ]:
signatures_RDD = (vectors_hash_RDD.map(lambda r: (r[0], np.sign(np.dot(r[1][0], r[1][1]))))
                    .map(lambda r: (r[0][0], (r[0][1], r[1])))
                    .groupByKey().mapValues(list)
                    .mapValues(sorted)
                    .map(lambda r: (r[0], [int(v[1]) for v in r[1]])))

Let's give look at a possibile signature for a movie.

In [ ]:
id, signature = signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

Now, `signatures_RDD` contains the so called *signature matrix*, that contains a signature for each movie in the dataset.

### Locality-sensitive hashing

It would be unthinkable to compare all possible pairs of movies to find similar ones among them. This would mean scanning all the rows in the signature matrix to compute the relative frequency between possible pairs of movies. So, we proceed by applying locality-sensitive hashing. In this approach we reduce the number of rows that determine the signature of a movie by hashing so called bands of rows. The rationale is that similar movies are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity. Looking at the resulting signatures we consider as a candidate pair only those for which their cosine similarity exceeds a threshold $t$.

We begin by dividing the signature matrix into $b$ bands of $r$ rows each. The choice of $r$ and $b$ depends on the threshold $t$ on the cosine distance between pairs of movies. The value of the threshold $t$ is approximately the value of similarity at which the probability of becoming a candidate is $\frac{1}{2}$.
So, we keep into account the following relationships (for brevity $l$=`signature_length`):   

\begin{equation*}
    \begin{cases}
        t=\left(\frac{1}{b}\right)^\frac{1}{r} \\
        r\cdot b = l
    \end{cases}
\end{equation*}

The function `band_size` solves this system in $r$ by computing it's value through
\begin{equation*}
    r=-\frac{W(-l\ln(t))}{\ln(t)}
\end{equation*}

where $W$ is the Lambert $W$ function, used to solve equations in the form $we^{w}=z$ for $w$.

In [ ]:
from scipy.special import lambertw

def band_size(t, sig_len):
    """ Compute the number of rows that form a band for the LSH technique

    Args:
        t (int): desired threshold in [0,1] on the Jaccard similarity between pairs of sets
        sig_len (int): signature length for the sets

    Returns:
        int: ideal number of rows contained in the band
    """
    return -math.ceil(lambertw(-sig_len*np.log(t)).real/np.log(t))

def r_b_choice(t,sig_len):
    """ Choose the adeguate number of rows a band and the number of bands for the LSH technique

    Args:
        t (int): desired threshold in [0,1] on the Jaccard similarity between pairs of sets
        sig_len (int): signature length for the sets

    Returns:
        int: ideal number of rows, and consequent number of bands on the basis of the signature length
    """
    r=band_size(t,signature_length)
    b=math.ceil(signature_length/r)
    return (r,b)

In [ ]:
t=0.99
t_bc = sc.broadcast(t)
r,b = r_b_choice(t,signature_length)
print("The chosen parameters are: \n r: {} \n b: {}".format(r,b))

Having chosen the parameters we proceed by subdividing the rows of the similarity matrix into bands.

In [ ]:
def split_list(l,b):
    """ Split list into b lists of equal length

    Args:
        l (list): list of elements
        b (int): number of sublists

    Returns:
        list: list formed by b sublists all of the same length, except for the last one if len(l) is not a multiple of b
    """
    split_list = [(list(a)) for a in np.array_split(np.array(l), b)]
    return [split_list[i] for i in range(b)]

We proceed by hashing the rows in each of the $b$ bands for each vector. For each band we use a different bucket array, so that signatures with two equal vectors in separate bands are hashed differently. This is done by setting a hash function with a distinct seed for all bands.

In [ ]:
def RDD_LSH(signatures_rdd, b, hash_bucket_size):
    """ Compute the hashed signatures with the LSH technique

    Args:
        signatures_rdd (RDD): RDD of (set key, signature for set)
        b (int): number of bands in which to split the signature matrix represented by signatures_rdd
        hash_bucket_size (int): the size of the bucket to which the signature portions in each band get hashed to

    Returns:

    """
    b_bc = sc.broadcast(b)
    hash_bucket_size_bc = sc.broadcast(hash_bucket_size)
    return (signatures_rdd
            .map(lambda r: (r[0],split_list(r[1],b_bc.value)))
            .map(lambda r: (r[0],[hash_object(bytes(str(t),'ascii'),i,hash_bucket_size_bc.value) for i,t in enumerate(r[1])])))

Also, to avoid hashing distinct portions of a signature in the same bucket it is important to choose a great enough bucket. Here we have evaluted the number of tuples given by $\{-1,1\}^r$.

In [ ]:
hash_bucket_size = 2**20
hashed_signatures_RDD = RDD_LSH(signatures_RDD, b, hash_bucket_size).cache()

Let's give a look at the new compact representation of a movie.

In [ ]:
id, signature = hashed_signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

### Find similar items

Now we search for candidate pairs among the reviews. We consider as possible similar couples of reviews those that have Jaccard similarity at least $t$.

In [ ]:
def RDD_pairs(rdd):
    """ Put together all possible pairs of rows, without repetitions

    Args:
        rdd (RDD): RDD of (row key, information regarding row)

    Returns:
        RDD: RDD of ((r1,r2),(info1, info2)), with r1>r2 to avoid having duplicates
    """
    pairs_RDD = rdd.cartesian(rdd).filter(lambda r: r[0][0] > r[1][0])
    return pairs_RDD.map(lambda r: ((r[0][0], r[1][0]),(r[0][1], r[1][1])))

def cosine_distance(vec1,vec2):
    """
    Compute the cosine distance between two vectors.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.

    Returns:
        float: The cosine distance between the two vectors, in radians.
    """
    cosine = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return np.arccos(cosine)

def cosine_similarity(vec1,vec2):
    """
    Compute the cosine similarity between two vectors.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.

    Returns:
        float: The cosine similarity between the two vectors.
    """

    return (math.pi-cosine_distance(vec1,vec2))/math.pi

def RDD_similar_items(rdd,t, similarity_function):
    """ Filter from pairs of rows whether they have similarity at least t

    Args:
        rdd (RDD): RDD of (row key, information regarding the row)
        t (int): desired threshold on the Jaccard similarity
        similarity_function (function): function to compute the similarity between two vectors

    Returns:
        RDD: RDD of ((k1,k2),1) if similarity_function(info1,info2)>=t
    """
    t_bc = sc.broadcast(t)
    pairs_RDD = RDD_pairs(rdd)
    return (pairs_RDD
            .filter(lambda r: similarity_function(r[1][0],r[1][1])>=t_bc.value)
            .map(lambda r: (r[0],similarity_function(r[1][0],r[1][1]))))

The potential candidate pairs are the following.

In [ ]:
candidate_pairs_LSH_RDD = RDD_similar_items(hashed_signatures_RDD,t, cosine_similarity).cache()
candidate_pairs = candidate_pairs_LSH_RDD.collect()

In [ ]:
len(candidate_pairs)

In [ ]:
movies_RDD.filter(lambda r: r[0]==candidate_pairs[0][0][0]).collect()

In [ ]:
movies_RDD.filter(lambda r: r[0]==candidate_pairs[0][0][1]).collect()

In [ ]:
candidate_pairs[:20]